# Nazario preparation for Dataset 1

This notebook prepares `nazario.jsonl` for the v2 Core human-side gathering stage.

The input file is already pre-parsed (body extracted, headers stripped), so the main work here
is normalization, URL masking, language filtering, deduplication, and Core schema field assignment.

Per `v2/docs/dataset_design_final.md` §5.5:
- `scenario_family = phishing_email`
- `channel = email`, `fraudness = fraud`
- All records included after cleaning and quality filtering (no Core policy exclusion)
- `time_band` split: `legacy` (pre-2015 mbox sources) vs `modern` (`phishing-2015` through `phishing-2025`)

No subtype annotation — §5.5 says only "при возможности" (optional/diagnostic), not a preparation requirement.

Pipeline:
- normalize text (NFKC, whitespace, broken encoding),
- URL masking (two-pass to catch space-broken URLs),
- English-like language filter,
- outlier cap: token_length ≤ 5 000 (removes parsing artifacts; ~1–2 rows with 300k+ tokens),
- deduplication by masked text,
- quality gate (char_length ≥ 30),
- assign `time_band` from `source` field,
- assign all Core schema fields via `v2/config.py` for `length_bin`,
- export to `v2/data/interim/gathered/nazario_prepared.jsonl`.

**Audit fix (Apr-2026):** added outlier cap; `length_bin` now imported from `v2/config.py`
(thresholds unchanged: email short<100, medium<400, long≥400).

In [1]:
from __future__ import annotations

import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from langdetect import detect, LangDetectException, DetectorFactory

DetectorFactory.seed = 0  # reproducibility

BASE     = Path('/Users/askar/projects/antifraud-deepfake-detection/v2')
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin  # channel-level thresholds from v2/config.py
RAW_PATH = BASE / 'data/raw/collected/nazario.jsonl'
OUT_DIR  = BASE / 'data/interim/gathered'
OUT_PATH = OUT_DIR / 'nazario_prepared.jsonl'

URL_RE   = re.compile(r'(?i)(?:https?://\S+|www\.\S+|\b[a-z0-9][a-z0-9-]*\.[a-z]{2,}(?:/\S*)?)')
TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)

# Sources whose emails are all pre-2015 (legacy era)
LEGACY_SOURCES = {
    '20051114.mbox', 'phishing0.mbox', 'phishing1.mbox', 'phishing2.mbox', 'phishing3.mbox'
}
# phishing-YYYY sources: YYYY >= 2015 → modern
MODERN_SOURCE_RE = re.compile(r'^phishing-(\d{4})$')


def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\uFFFD', ' ')
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def mask_url(text: str) -> str:
    # First pass: catch full URLs (http://domain/path, www.domain, bare domain)
    text = URL_RE.sub('[URL]', text)
    # Second pass: catch bare http(s):// fragments left when URL was space-broken
    text = re.sub(r'https?://\S*', '[URL]', text)
    return text


def is_english(text: str) -> bool:
    """Detect English via langdetect (seed=0 for reproducibility)."""
    if not text or len(text.strip()) < 15:
        return False
    try:
        return detect(text[:2000]) == 'en'
    except LangDetectException:
        return False


def token_length(text: str) -> int:
    return len(TOKEN_RE.findall(text))


def time_band_from_source(source: str) -> str:
    source = (source or '').strip()
    if source in LEGACY_SOURCES:
        return 'legacy'
    m = MODERN_SOURCE_RE.match(source)
    if m and int(m.group(1)) >= 2015:
        return 'modern'
    return 'legacy'

In [2]:
# Load, preprocess, language filter, deduplicate
df = pd.read_json(RAW_PATH, lines=True, dtype=str)

df['text_norm']   = df['text'].map(normalize_text)
df = df[df['text_norm'] != ''].copy()

# URL-masked text = final output text field
df['text_masked'] = df['text_norm'].map(mask_url).map(normalize_text)
df = df[df['text_masked'] != ''].copy()

df['is_english'] = df['text_masked'].map(is_english)
df_en = df[df['is_english']].copy()

# Quality gate: minimum character length after masking
df_en = df_en[df_en['text_masked'].str.len() >= 30].copy()

# Outlier cap: remove parsing artifacts (e.g. one record with ~308k tokens).
# Legitimate phishing emails do not exceed 5 000 tokens.
TOKEN_RE_LOCAL = re.compile(r'\w+|[^\w\s]', re.UNICODE)
df_en = df_en[df_en['text_masked'].map(lambda t: len(TOKEN_RE_LOCAL.findall(t))) <= 5000].copy()

# Deduplicate by masked text (keep first occurrence, track duplicate count)
grp      = df_en.groupby('text_masked', sort=False)
prepared = grp.first().reset_index(drop=False)
prepared['n_duplicate_rows'] = grp.size().values

print(f'raw rows:             {len(pd.read_json(RAW_PATH, lines=True))}')
print(f'after text cleaning:  {len(df)}')
print(f'after english filter: {len(df_en)}')
print(f'after dedup:          {len(prepared)}')

raw rows:             8472
after text cleaning:  8157
after english filter: 7657
after dedup:          5201


In [3]:
# Assign Core schema fields and write output
prepared['text']         = prepared['text_masked']
prepared['char_length']  = prepared['text'].str.len()
prepared['token_length'] = prepared['text'].map(token_length)
prepared['length_bin']   = prepared['token_length'].map(lambda t: compute_length_bin(t, 'email'))
prepared['time_band']    = prepared['source'].map(time_band_from_source)

prepared['label']            = 0
prepared['label_str']        = 'human'
prepared['fraudness']        = 'fraud'
prepared['channel']          = 'email'
prepared['scenario_family']  = 'phishing_email'
prepared['source_family']    = 'nazario'
prepared['dataset_source']   = 'nazario.jsonl'
prepared['origin_model']     = 'human'
prepared['split']            = 'tbd'
prepared['lang_filter_method'] = 'english_like_ascii_v1'

# Preserve original metadata (rename 'from' → 'from_addr' to avoid Python keyword clash)
prepared['from_addr'] = prepared['from'].astype(str)
prepared['date_raw']  = prepared['date'].astype(str)
prepared['has_url']   = prepared['has_url'].map(
    lambda v: 'yes' if str(v).lower() in ('true', '1', 'yes') else 'no'
)
prepared['url_count'] = prepared['url_count'].fillna(0).astype(int)

out_cols = [
    'text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family',
    'source_family', 'dataset_source', 'time_band', 'length_bin',
    'char_length', 'token_length', 'has_url', 'url_count',
    'subject', 'from_addr', 'date_raw', 'source',
    'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows',
]
out_df = prepared[out_cols].copy()

OUT_DIR.mkdir(parents=True, exist_ok=True)
out_df.to_json(OUT_PATH, orient='records', lines=True, force_ascii=False)

print(f'written rows:         {len(out_df)}')
print(f'output:               {OUT_PATH}')

print('\ntime_band distribution:')
print(out_df['time_band'].value_counts())

print('\nlength_bin distribution:')
print(out_df['length_bin'].value_counts())

print('\nhas_url distribution:')
print(out_df['has_url'].value_counts())

written rows:         5201
output:               /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/nazario_prepared.jsonl

time_band distribution:
time_band
legacy    2609
modern    2592
Name: count, dtype: int64

length_bin distribution:
length_bin
medium    2842
short     1614
long       745
Name: count, dtype: int64

has_url distribution:
has_url
no     2805
yes    2396
Name: count, dtype: int64


In [4]:
# Validation and sanity checks
check = pd.read_json(OUT_PATH, lines=True)

assert check['text'].astype(str).str.strip().ne('').all(), 'empty texts found'
assert (check['label'] == 0).all(), 'unexpected label values'
assert (check['channel'] == 'email').all(), 'unexpected channel'
assert (check['scenario_family'] == 'phishing_email').all(), 'unexpected scenario_family'
assert (check['fraudness'] == 'fraud').all(), 'unexpected fraudness'
assert set(check['time_band']).issubset({'legacy', 'modern'}), 'unexpected time_band values'
assert set(check['length_bin']).issubset({'short', 'medium', 'long'}), 'unexpected length_bin values'
assert check['text'].str.contains(r'https?://', regex=True).sum() == 0, \
    'some texts still contain unmasked URLs'

print('Validation passed.')
print(f'Rows in gathered JSONL: {len(check)}')
print(f'Columns: {list(check.columns)}')

print('\nSample texts per time_band (first 2 each):')
for band in sorted(check['time_band'].unique()):
    print(f'\n=== {band} ===')
    for t in check.loc[check['time_band'] == band, 'text'].head(2):
        print(' ', t[:200])

print('\nFiles in gathered/:')
for p in sorted(OUT_DIR.glob('*')):
    print(f'  - {p.name}')

Validation passed.
Rows in gathered JSONL: 5201
Columns: ['text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family', 'source_family', 'dataset_source', 'time_band', 'length_bin', 'char_length', 'token_length', 'has_url', 'url_count', 'subject', 'from_addr', 'date_raw', 'source', 'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows']

Sample texts per time_band (first 2 each):

=== legacy ===
  eBay Suspension Need Help? Dear valued eBay member, During our regularly scheduled account maintenance and verification procedures, we have detected a slight error in your billing information. This mi
  Dear LaSalle Member, As part of our continuing commitment to protect your account and to reduce the instance of fraud on our website, we are undertaking a period review of our member accounts. You are

=== modern ===
  Dear Customer, You have received this email because we believe that your amazon account has been recently compromised.. Click Here to update your account R